# Batch Download and Mosaic

Download tiles for a county, mosaic, and compute hillshade.

In [ ]:
import abovepy
import numpy as np
from pathlib import Path

## Search by county

In [ ]:
counties = abovepy.list_counties()
print(f"{len(counties)} counties available")
tiles = abovepy.search(county="Woodford", product="dem_phase3")
print(f"Found {len(tiles)} tiles for Woodford County")

## Download all tiles

In [ ]:
output_dir = Path("./data/woodford_dem")
paths = abovepy.download(tiles, output_dir=output_dir)
print(f"Downloaded {len(paths)} files")

## Create VRT mosaic

In [ ]:
vrt_path = output_dir / "mosaic.vrt"
result = abovepy.mosaic(paths, output=vrt_path)
print(f"VRT created: {result}")

## Compute hillshade

In [ ]:
import rasterio
import matplotlib.pyplot as plt

with rasterio.open(vrt_path) as src:
    elev = src.read(1)

dy, dx = np.gradient(elev)
slope = np.sqrt(dx**2 + dy**2)
aspect = np.arctan2(-dy, dx)
hillshade = np.sin(np.radians(45)) * np.cos(slope) + np.cos(np.radians(45)) * np.sin(slope) * np.cos(np.radians(315) - aspect)
hillshade = np.clip(hillshade * 255, 0, 255).astype(np.uint8)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 8))
ax1.imshow(elev, cmap='terrain')
ax1.set_title('DEM')
ax1.axis('off')
ax2.imshow(hillshade, cmap='gray')
ax2.set_title('Hillshade')
ax2.axis('off')
plt.tight_layout()
plt.show()